# 02 Jobs, Stages and Tasks

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Explain how one action becomes a <b>job</b>, how a job is split into <b>stages</b> at shuffle boundaries, and how each stage runs as <b>tasks</b> (one per partition). Predict the number of stages and tasks from a query plan.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Jobs, stages and tasks are the vocabulary of the Spark UI and of every performance conversation: "stage 3 has 2,000 tasks and one of them takes 40 minutes". If you can predict them from your code, you can find slow stages, skewed tasks and unnecessary shuffles. "What is the difference between a job, a stage and a task?" is a standard interview question.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The execution hierarchy</b><br>
<code>`</code><br>Application (one SparkSession / notebook)<br>└── Job          one per ACTION (count, collect, write, show ...)<br>    └── Stage    a chain of narrow work with no shuffle inside; a new stage starts after each shuffle<br>        └── Task one per PARTITION of the stage, runs on one core<br><code>`</code>
<ul><li><b>Stage boundaries</b> are the <code>Exchange</code> operators (shuffles) in the physical plan</li><li><b>Map-side</b> stages write shuffle files, <b>reduce-side</b> stages read them</li><li>Tasks in the first stage = number of input partitions (files/splits). Tasks after a shuffle = number of shuffle partitions (<code>spark.sql.shuffle.partitions</code>, adjusted by AQE)</li></ul>
</div>

```
orders.filter(...).groupBy("country").count().show()

Job 1 (triggered by show)
├── Stage 1: scan → filter → partial count   (one task per input partition, e.g. 8)
│        ── shuffle write ──
└── Stage 2: read shuffle → final count       (one task per shuffle partition, e.g. 200 → AQE → 1)
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A job shows 3 stages in the Spark UI. Stage 1 (reading 2,000 files) has 2,000 tasks and finishes in 2 minutes. Stage 3 (after a join) has 200 tasks, 199 finish in seconds and one runs for 35 minutes. That pattern, one task much slower than its siblings, is <b>data skew</b> on the join key, and it's visible only because the engineer knew to read the UI at the stage and task level.
</div>

## Syntax: how to see jobs, stages and tasks

| Where | What you see |
|---|---|
| `df.explain()` | `Exchange` = stage boundary, `*(n)` = code-generated pipeline inside a stage |
| Spark UI → Jobs tab (classic compute) | One row per job, with its stages |
| Spark UI → Stages tab | Tasks per stage, durations, shuffle read/write sizes |
| Query profile ("See performance" under a cell, serverless) | Operators, rows, time and shuffle bytes per operator |

## 1. Count stages from the plan

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds three queries with 0, 1 and 2 shuffles and counts the <code>Exchange</code> operators in each plan.<br><br>
<b>Why it matters</b><br><b>Stages = exchanges + 1</b> for a simple linear query. This rule lets you predict the stage count before running anything.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>narrow only</code>: 0 exchanges (1 stage). <code>groupBy</code>: 1 exchange (2 stages). <code>groupBy</code> then <code>orderBy</code>: 2 exchanges (3 stages).
</div>

In [0]:
import io, contextlib, re
from pyspark.sql import functions as F

def plan_text(df):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

def count_exchanges(df):
    return len(re.findall(r"Exchange ", plan_text(df)))

data = spark.range(0, 1_000_000, 1, numPartitions=8).withColumn("country", (F.col("id") % 5).cast("string"))

narrow = data.filter("id % 2 = 0").select("id", "country")
one_shuffle = data.groupBy("country").count()
two_shuffles = data.groupBy("country").count().orderBy(F.desc("count"))

for name, df in [("narrow only", narrow), ("groupBy", one_shuffle), ("groupBy + orderBy", two_shuffles)]:
    ex = count_exchanges(df)
    print(f"{name:<20} exchanges: {ex}  -> stages: {ex + 1}")

## 2. Tasks per stage = partitions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Counts the partitions of the first stage (the input) and of the stage after the shuffle, with the shuffle partition setting at 8.<br><br>
<b>Why it matters</b><br>The first stage has one task per input partition. The stage after a shuffle has one task per shuffle partition. Here we read partitions with <code>spark_partition_id()</code>, which works on serverless where the Spark UI isn't available.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>spark.sql.shuffle.partitions</code> is one of the settings you <b>can</b> change on serverless. It is often set to <code>auto</code> there, which lets Databricks choose.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Input stage: 8 partitions (8 tasks). After the shuffle with AQE turned off: up to 8 partitions, but only the ones that receive keys are non-empty (5 countries hash into at most 5 of them).
</div>

In [0]:
def non_empty_partitions(df):
    return df.select(F.spark_partition_id().alias("p")).distinct().count()

print("input stage tasks:", non_empty_partitions(data))

original = spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.shuffle.partitions", "8")
try:
    spark.conf.set("spark.sql.adaptive.enabled", "false")      # may be rejected on serverless
    aqe_note = "AQE off"
except Exception:
    aqe_note = "AQE setting not changeable here"
print(f"post-shuffle partitions holding data ({aqe_note}):", non_empty_partitions(data.groupBy("country").count()))

try:
    spark.conf.set("spark.sql.adaptive.enabled", "true")
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", original)

## 3. One action, one job (usually)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs the same DataFrame with three different actions and times each.<br><br>
<b>Why it matters</b><br>Each action is a <b>separate job</b> that re-runs the whole plan from the source (unless data is cached or written). Calling <code>count()</code> and then <code>show()</code> on an expensive DataFrame does the work twice.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three timings of the same order of magnitude: each action recomputed the aggregation. A few operations can trigger more than one job (for example schema inference or sampling for a sort), so job counts in the UI can be slightly higher than the number of actions.
</div>

In [0]:
import time

expensive = (
    spark.range(0, 3_000_000, 1, numPartitions=8)
    .withColumn("bucket", F.col("id") % 1000)
    .groupBy("bucket").agg(F.sum("id").alias("total"))
)

for action_name, action in [("count", lambda: expensive.count()),
                            ("collect", lambda: expensive.collect()),
                            ("first", lambda: expensive.orderBy("bucket").first())]:
    start = time.time()
    action()
    print(f"{action_name:<8} job took {time.time() - start:.2f}s")

## 4. Reading a plan as stages

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the formatted plan of a join plus aggregation and maps each operator to its stage.<br><br>
<b>Why it matters</b><br>In real debugging you look at a plan (or the query profile) and ask "which stage is this operator in, and which stage is slow?". Exchanges separate the stages, and the <code>*(n)</code> codegen IDs group the operators that run together inside a stage.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A plan with a sort-merge join (both sides shuffled) and an aggregation. Reading it bottom-up gives: stage 1 scans and shuffles the left side, stage 2 scans and shuffles the right side, stage 3 sort-merges, joins and does the partial aggregate, and stage 4 does the final aggregate.
</div>

In [0]:
left = spark.range(0, 1_000_000, 1, numPartitions=8).withColumn("key", F.col("id") % 10_000)
right = spark.range(0, 10_000, 1, numPartitions=4).withColumnRenamed("id", "key").withColumn("region", F.col("key") % 7)

query = (
    left.join(right.hint("merge"), "key")           # force a sort-merge join for the demo
    .groupBy("region").agg(F.count("*").alias("rows"))
)
query.explain("formatted")

## 5. Tasks run in waves

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains what happens when a stage has more tasks than cores, with a small calculation.<br><br>
<b>Why it matters</b><br>With 200 tasks and 40 cores, tasks run in <b>5 waves</b>. If one task in the last wave is slow, the whole stage waits for it. That's why both too few partitions (idle cores) and skewed partitions (one long task) hurt.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A small table of task counts, cores and the resulting number of waves.
</div>

In [0]:
import math

for tasks, cores in [(8, 32), (200, 40), (1000, 64), (2000, 16)]:
    print(f"{tasks:>5} tasks on {cores:>3} cores -> {math.ceil(tasks / cores):>4} wave(s), idle cores in last wave: {cores * math.ceil(tasks / cores) - tasks}")

## Under the hood

```
ACTION  ──▶  DAGScheduler                           TaskScheduler                Executors
             - walk the physical plan backwards     - one TaskSet per stage     - run tasks
             - cut a new stage at every Exchange    - schedule tasks on free    - write shuffle files (map side)
             - submit parent stages first             cores, near their data    - fetch shuffle blocks (reduce side)
             - stage N+1 starts when all of stage    - retry failed tasks
               N's shuffle output is written
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> each action submits one job (sometimes a few small extra jobs, for example for sampling).

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> each shuffle ends a stage. Map tasks write shuffle files, then reduce tasks of the next stage read them.

**Skipped stages:** if the shuffle output of a stage still exists from an earlier job, the Spark UI shows the stage as "skipped" and reuses its output. **Speculative execution** (classic compute) can launch a backup copy of a very slow task on another executor.

The plan below has three exchanges, so four stages. Count them yourself.

In [0]:
(
    spark.range(0, 1_000_000, 1, numPartitions=8)
    .repartition(16)                                   # exchange 1
    .groupBy((F.col("id") % 100).alias("k")).count()   # exchange 2
    .orderBy("k")                                      # exchange 3
).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: the same data is processed several times</b><br>
Several actions on the same expensive DataFrame. Write the intermediate result to a table, or restructure to use one action.<br><br>
<b>⛔ Problem: a stage with thousands of tiny tasks</b><br>Too many input files or shuffle partitions. Compact files, lower <code>spark.sql.shuffle.partitions</code>, or rely on AQE coalescing.<br><br>
<b>⛔ Problem: one task takes much longer than the others</b><br>Data skew: one partition holds far more data. See <code>05_optimization/02_data_skew</code>.<br><br>
<b>⛔ Problem: can't find the Spark UI on serverless</b><br>Use the query profile ("See performance" under the cell output, or the Query History page).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between a job, a stage and a task?</b><br>
A job is created for each action. It's split into stages at shuffle boundaries, where each stage is a set of narrow transformations that can run without moving data. Each stage runs as tasks, one per partition, and each task runs on one core.<br><br>

<b>🎤 2. What creates a new stage?</b><br>
A shuffle, which is an <code>Exchange</code> in the physical plan. Operations such as <code>groupBy</code>, <code>join</code> (non-broadcast), <code>distinct</code>, <code>repartition</code> and <code>orderBy</code> need one.<br><br>

<b>🎤 3. How many tasks does a stage have?</b><br>
One per partition. The first stage has one task per input split, and a stage after a shuffle has one per shuffle partition, which AQE may coalesce.<br><br>

<b>🎤 4. What happens when there are more tasks than cores?</b><br>
Tasks run in waves. Each core takes the next task when it finishes one, and the stage ends when the slowest task finishes.<br><br>

<b>🎤 5. Why might calling <code>count()</code> and then <code>show()</code> be slow?</b><br>
Each action is a separate job that recomputes the plan from the source, unless the data is cached or materialized.<br><br>

<b>🎤 6. What is a skipped stage?</b><br>
A stage whose shuffle output already exists from a previous job, so Spark reuses it instead of recomputing it.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. In the Spark UI, a stage has 200 tasks. 199 tasks finish in about 10 seconds, and one task runs for 25 minutes. What is the most likely cause?</b><br>
A. Too many executors<br>
B. Data skew, where one partition holds far more data than the others<br>
C. The driver is out of memory<br>
D. Column pruning is disabled<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. What determines the boundary between two stages in a Spark job?</b><br>
A. Each transformation<br>
B. Each action<br>
C. A shuffle (exchange) of data between partitions<br>
D. Each Python cell<br>
<details><summary><b>Show answer</b></summary><b>C.</b> Actions create jobs. Shuffles split jobs into stages.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a query with exactly 2 shuffles and confirm it with <code>count_exchanges</code></li><li>Predict the number of input tasks of <code>spark.range(0, 10_000, 1, numPartitions=7).groupBy(...)</code>, then check with <code>non_empty_partitions</code></li><li>Time <code>count()</code> twice on the same expensive DataFrame. Why is the second run not instant?</li><li>In the <code>query</code> plan from section 4, which operators run in the same stage as <code>SortMergeJoin</code>?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. distinct (shuffle 1) + orderBy (shuffle 2)
q = data.select("country").distinct().orderBy("country")
print("exchanges:", count_exchanges(q))

# 2. Input tasks = 7 partitions
print("input tasks:", non_empty_partitions(spark.range(0, 10_000, 1, numPartitions=7)))

# 3. No caching: each action is a new job that recomputes the aggregation
for i in range(2):
    start = time.time(); expensive.count(); print(f"count #{i + 1}: {time.time() - start:.2f}s")

# 4. Sort (both sides), SortMergeJoin, Project and the partial HashAggregate run in the same stage,
#    between the two input exchanges and the exchange before the final aggregate.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Action → job → stages (cut at every shuffle) → tasks (one per partition)</li><li>Stages ≈ exchanges + 1 for a linear query. Read <code>Exchange</code> in <code>explain()</code></li><li>Input stage tasks = input partitions. Post-shuffle tasks = shuffle partitions (AQE may coalesce)</li><li>More tasks than cores → waves. The slowest task decides the stage time</li><li>Each action recomputes the plan unless the result is materialized</li><li>Serverless: use the query profile instead of the Spark UI</li></ul>
</div>

| Task | Code / place |
|---|---|
| Stage boundaries | `df.explain()` → `Exchange` |
| Detailed plan | `df.explain("formatted")` |
| Partitions holding data | `df.select(F.spark_partition_id()).distinct().count()` |
| Shuffle partitions | `spark.conf.set("spark.sql.shuffle.partitions", "64")` |
| Per-task metrics | Spark UI → Stages (classic), query profile (serverless) |

## Git commit

```
git add 02_spark_internals/02_jobs_stages_tasks.ipynb
git commit -m "add 02_02 jobs stages tasks notebook"
```